In [25]:
#import modules 
import pandas as pd
import numpy as np
from typing import List
from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors

RANDOM_STATE = 42

## CLEANING THE DATA

## Calculate 2D and 3D descriptors

In [26]:
def calculate_all_descriptors(smiles_list: List[str], ignore_3D: bool = False) -> pd.DataFrame:
    """
    Calculate all 1826 Mordred molecular descriptors for a list of SMILES.
    2D: 1613 descriptors, 3D: 213 descriptors (requires conformer generation).
    """
    mols = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol and not ignore_3D:
            mol = Chem.AddHs(mol)
            params = AllChem.ETKDGv3()
            params.randomSeed = RANDOM_STATE
            if AllChem.EmbedMolecule(mol, params) != -1:
                try:
                    AllChem.MMFFOptimizeMolecule(mol)
                except Exception:
                    pass
        mols.append(mol)
    
    calc = Calculator(descriptors, ignore_3D=ignore_3D)
    df = calc.pandas(mols)
    df.insert(0, 'SMILES', smiles_list)
    
    # Convert error values to NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    df['SMILES'] = smiles_list
    
    return df

In [27]:
#Example usage
smiles = ['CCO', 'c1ccccc1', 'CC(=O)O']
tmp_df = calculate_all_descriptors(smiles)
print(tmp_df.shape)  # (3, ~1824) including SMILES column

100%|██████████| 3/3 [00:01<00:00,  1.89it/s]


(3, 1827)


In [28]:
# Load the pre-calculated descriptors from a CSV file
df = pd.read_excel('../Data/hob_data_set.xlsx', sheet_name='Training set')
display(df.head())

,Name,smile,value,label_cutoff_50%,our model predition_50%,label_cutoff_20%,our model predition_20%
0,sulfadiazine,Nc1ccc(S(=O)(=O)Nc2ncccn2)cc1,90,1,1,1.0,1.0
1,clofarabine,Nc1nc(Cl)nc2c1ncn2C1OC(CO)C(O)C1F,50,1,1,1.0,1.0
2,sulfamethoxazole,Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1,99,1,1,1.0,1.0
3,tolazoline,c1ccc(CC2=NCCN2)cc1,90,1,1,1.0,1.0
4,cotinine,CN1C(=O)CCC1c1cccnc1,97,1,1,1.0,1.0


In [29]:
# Select only the relevant columns and calculate descriptors for the SMILES strings
df = df[["smile","label_cutoff_50%","label_cutoff_20%"]]
descriptors_df = calculate_all_descriptors(df["smile"].tolist()) 

# Save the raw descriptors to a CSV file
descriptors_df.to_csv('../Data/mordred_descriptors_RAW.csv', index=False)
print(f"Raw descriptors saved: {descriptors_df.shape}")

[08:46:33] UFFTYPER: Unrecognized charge state for atom: 1
100%|██████████| 1157/1157 [01:26<00:00, 13.36it/s]


Raw descriptors saved: (1157, 1827)


## Calculate the exact number of NaN Group

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset
raw_df = pd.read_csv('../Data/mordred_descriptors_RAW.csv')
print(f"Raw shape: {raw_df.shape}")

# Calculate NaN count per descriptor (excluding SMILES)
X_raw = raw_df.drop(columns=['SMILES'])
nan_counts = X_raw.isnull().sum()
total_mols = len(X_raw)

# Category classification
always_missing     = nan_counts[nan_counts == total_mols]
above_90           = nan_counts[(nan_counts > total_mols * 0.90) & (nan_counts < total_mols)]
between_50_90      = nan_counts[(nan_counts > total_mols * 0.50) & (nan_counts <= total_mols * 0.90)]
between_20_50      = nan_counts[(nan_counts > total_mols * 0.20) & (nan_counts <= total_mols * 0.50)]
between_5_20       = nan_counts[(nan_counts > total_mols * 0.05) & (nan_counts <= total_mols * 0.20)]
below_5            = nan_counts[(nan_counts > 0) & (nan_counts <= total_mols * 0.05)]
no_missing         = nan_counts[nan_counts == 0]

print("\n=== NaN Summary ===")
print(f"Total descriptors (excl. SMILES): {len(nan_counts)}")
print(f"No missing values:                {len(no_missing)}")
print(f"100% NaN (always missing):        {len(always_missing)}")
print(f">90% NaN (almost always missing): {len(above_90)}")
print(f"50-90% NaN:                       {len(between_50_90)}")
print(f"20-50% NaN:                       {len(between_20_50)}")
print(f"5-20% NaN:                        {len(between_5_20)}")
print(f"<5% NaN:                          {len(below_5)}")
print(f"\nTotal with any NaN: {(nan_counts > 0).sum()}")
print(f"Total NaN values:   {nan_counts.sum()}")

Raw shape: (1157, 1827)

=== NaN Summary ===
Total descriptors (excl. SMILES): 1826
No missing values:                1499
100% NaN (always missing):        76
>90% NaN (almost always missing): 40
50-90% NaN:                       38
20-50% NaN:                       14
5-20% NaN:                        6
<5% NaN:                          153

Total with any NaN: 327
Total NaN values:   176453


In [31]:
summary_data = {
    'NaN Group': [
        'No missing values',
        '100% NaN (Always missing)',
        '>90% NaN (Almost always missing)',
        '50–90% NaN (Structural absence)',
        '20–50% NaN (Structural absence)',
        '5–20% NaN (Structural absence)',
        '<5% NaN (Calculation failure)'
    ],
    'N Descriptors': [
        len(no_missing),
        len(always_missing),
        len(above_90),
        len(between_50_90),
        len(between_20_50),
        len(between_5_20),
        len(below_5)
    ],
    'Likely Cause': [
        'Clean descriptor — no missing data',
        'Exotic atoms absent in all molecules (Pb, Sn, Si, etc.)',
        'Rare atom types absent in >90% of drug-like molecules',
        '2D atom-type descriptor — chemical motif absent in molecule',
        '2D atom-type descriptor — chemical motif absent in molecule',
        '2D atom-type descriptor — chemical motif absent in molecule',
        'Isolated conformer generation failure or calculation instability'
    ],
    'Decision': [
        'Keep',
        'Drop',
        'Drop',
        'Keep (fill 0) / Drop (low correlation with target)',
        'Keep (fill 0) / Drop (low correlation with target)',
        'Fill with 0',
        'Median imputation (training set medians only)'
    ],
    'Justification': [
        '—',
        'Zero information content across all molecules',
        'Near-zero variance — negligible signal',
        'NaN = genuine absence of atom type → 0 is chemically correct. Low-correlation descriptors dropped via Point Biserial (|r|>0.10, p<0.05)',
        'Same as above',
        'Chemically verified: NaN confirmed to reflect absent atom type (e.g. aromatic CH)',
        'NaN reflects calculation failure, not absence — median of observed values is most neutral estimate'
    ]
}

import pandas as pd
table_df = pd.DataFrame(summary_data)
print(table_df.to_string(index=False))

                       NaN Group  N Descriptors                                                     Likely Cause                                           Decision                                                                                                                           Justification
               No missing values           1499                               Clean descriptor — no missing data                                               Keep                                                                                                                                       —
       100% NaN (Always missing)             76          Exotic atoms absent in all molecules (Pb, Sn, Si, etc.)                                               Drop                                                                                           Zero information content across all molecules
>90% NaN (Almost always missing)             40            Rare atom types absent in >90% of drug-li

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Missing Values Summary"

# Colors
header_fill = PatternFill(start_color="2E4057", end_color="2E4057", fill_type="solid")
green_fill  = PatternFill(start_color="E6F7F0", end_color="E6F7F0", fill_type="solid")
red_fill    = PatternFill(start_color="FDE8E8", end_color="FDE8E8", fill_type="solid")
amber_fill  = PatternFill(start_color="FDF4E0", end_color="FDF4E0", fill_type="solid")
blue_fill   = PatternFill(start_color="E8F1FE", end_color="E8F1FE", fill_type="solid")
thin = Side(style='thin', color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# Headers
headers = ['NaN Group', 'N Descriptors', 'Likely Cause', 'Decision', 'Justification']
for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_idx, value=h)
    cell.font = Font(bold=True, color="FFFFFF", size=11)
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = border


fills = [green_fill, red_fill, red_fill, amber_fill, amber_fill, amber_fill, blue_fill]
for row_idx, (fill, i) in enumerate(zip(fills, range(7)), 2):
    values = [summary_data['NaN Group'][i], summary_data['N Descriptors'][i],
              summary_data['Likely Cause'][i], summary_data['Decision'][i],
              summary_data['Justification'][i]]
    for col_idx, val in enumerate(values, 1):
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.fill = fill
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = border
        if col_idx == 2:
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.font = Font(bold=True, size=11)

# Column widths
widths = [22, 14, 40, 28, 55]
for col_idx, width in enumerate(widths, 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = width

# Row heights
ws.row_dimensions[1].height = 30
for row_idx in range(2, 9):
    ws.row_dimensions[row_idx].height = 55

# Title row above headers
ws.insert_rows(1)
ws.merge_cells('A1:E1')
title_cell = ws['A1']
title_cell.value = f'Missing Value Handling Strategy — HOB Molecular Descriptors (n={total_mols})'
title_cell.font = Font(bold=True, size=12, color="2E4057")
title_cell.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 35

wb.save('../Data/missing_values_table.xlsx')
print("Table saved to missing_values_table.xlsx")

Table saved to missing_values_table.xlsx


In [33]:
display(descriptors_df.head())

,SMILES,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,Nc1ccc(S(=O)(=O)Nc2ncccn2)cc1,13.093540,11.115492,0,0,21.139069,2.370239,4.740478,21.139069,1.243475,...,9.645817,49.622129,250.052447,9.261202,536,23,86.0,97.0,5.895833,3.708333
1,Nc1nc(Cl)nc2c1ncn2C1OC(CO)C(O)C1F,15.710828,14.057406,0,0,25.397628,2.520227,4.881224,25.397628,1.269881,...,10.082679,70.336447,303.053445,9.775918,753,34,110.0,135.0,7.500000,4.361111
2,Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1,13.202929,11.709699,0,0,20.712705,2.383498,4.742568,20.712705,1.218394,...,9.663770,63.242131,253.052112,9.037575,535,22,88.0,100.0,6.506944,3.625000
3,c1ccc(CC2=NCCN2)cc1,9.192388,8.248586,0,2,15.982234,2.246428,4.436582,15.982234,1.331853,...,8.907477,55.114662,160.100048,6.670835,209,11,58.0,64.0,2.722222,2.750000
4,CN1C(=O)CCC1c1cccnc1,9.996954,9.253985,0,0,16.886871,2.383381,4.647225,16.886871,1.298990,...,9.339525,58.137510,176.094963,7.043799,238,17,66.0,77.0,4.194444,2.916667


In [34]:
# Check for non-numeric columns and count unique values
for col in descriptors_df.columns:
    if descriptors_df[col].dtype == 'object':
        print(f"{col}: {descriptors_df[col].nunique()} unique values")

SMILES: 1157 unique values


In [35]:
# Identify non-numeric columns by checking which columns are not included in the describe() output
dsc = descriptors_df.describe()
for col in descriptors_df.columns:
    if col not in dsc.columns:
        print(f"{col} is not numeric.")

SMILES is not numeric.
Lipinski is not numeric.
GhoseFilter is not numeric.


In [36]:
# Convert Lipinski and GhoseFilter to integers (0/1) cause they are boolean
descriptors_df['Lipinski'] = descriptors_df['Lipinski'].astype(int)
descriptors_df['GhoseFilter'] = descriptors_df['GhoseFilter'].astype(int)
display (descriptors_df['GhoseFilter'].value_counts())
display (descriptors_df['Lipinski'].value_counts())

GhoseFilter
1    857
0    300
Name: count, dtype: int64

Lipinski
1    947
0    210
Name: count, dtype: int64

In [37]:
# Check for NaN values in the descriptors and count how many descriptors have missing values
nan_summary = descriptors_df.isnull().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)
print(f"Total descriptors with NaN values: {len(nan_summary)}")
print(f"Total NaN values in dataset: {descriptors_df.isnull().sum().sum()}")
print(nan_summary)

Total descriptors with NaN values: 327
Total NaN values in dataset: 176453
MINssssPb    1157
MINsPbH3     1157
MINssPbH2    1157
MINsssPbH    1157
MINsssssP    1157
             ... 
AATS6p          6
AATS6dv         6
Vabc            6
AXp-4d          2
AXp-4dv         2
Length: 327, dtype: int64


In [38]:
# See how many descriptors are completely empty
all_nan = descriptors_df.isnull().sum()
completely_empty = (all_nan == len(descriptors_df)).sum()
print(f"Descriptors with NaN in ALL molecules: {completely_empty}")

# And what percentage of your 327 they represent
print(f"That's {completely_empty/len(nan_summary)*100:.1f}% of your problematic descriptors")

Descriptors with NaN in ALL molecules: 76
That's 23.2% of your problematic descriptors


In [39]:
# Look at the ones that are partially empty and see how bad the problem is
descriptors_df.MINssssPb

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
1152   NaN
1153   NaN
1154   NaN
1155   NaN
1156   NaN
Name: MINssssPb, Length: 1157, dtype: float64

In [40]:
# Look at the ones that are partially empty and see how bad the problem is
partial_nan = nan_summary[(nan_summary > 0) & (nan_summary < len(descriptors_df))]

# Bucket them by severity
print("NaN in >50% of molecules:", (partial_nan > len(descriptors_df)*0.50).sum())
print("NaN in 20-50% of molecules:", ((partial_nan > len(descriptors_df)*0.20) & (partial_nan <= len(descriptors_df)*0.50)).sum())
print("NaN in 5-20% of molecules:", ((partial_nan > len(descriptors_df)*0.05) & (partial_nan <= len(descriptors_df)*0.20)).sum())
print("NaN in <5% of molecules:", (partial_nan <= len(descriptors_df)*0.05).sum())

NaN in >50% of molecules: 78
NaN in 20-50% of molecules: 14
NaN in 5-20% of molecules: 6
NaN in <5% of molecules: 153


## Descriptors with NaN in ALL molecules

In [41]:
# Drop descriptors with NaN in ALL molecules
all_nan_cols = nan_summary[nan_summary == len(descriptors_df)].index.tolist()

print(f"Descriptors to drop: {len(all_nan_cols)}")
descriptors_df = descriptors_df.drop(columns=all_nan_cols)
print(f"Remaining descriptors: {descriptors_df.shape[1]}")

Descriptors to drop: 76
Remaining descriptors: 1751


## NaN in >50% of molecules

In [42]:
investigate_50 = partial_nan[partial_nan > len(descriptors_df)*0.5]
print(investigate_50)

MAXsssNH    1156
MAXsssB     1156
MINsssNH    1156
MINsssB     1156
MINssNH2    1156
            ... 
MDEO-12      656
MAXssO       618
MINssO       618
MINsssN      613
MAXsssN      613
Length: 78, dtype: int64


In [43]:
borderline_nan = investigate_50[(investigate_50 > len(descriptors_df) * 0.50) & 
                                (investigate_50 <= len(descriptors_df) * 1)]

print(f"Borderline descriptors (50-90% NaN): {len(borderline_nan)}")
print(borderline_nan)
print(f"Total borderline descriptors (50-90% NaN): {len(borderline_nan)}")

Borderline descriptors (50-90% NaN): 78
MAXsssNH    1156
MAXsssB     1156
MINsssNH    1156
MINsssB     1156
MINssNH2    1156
            ... 
MDEO-12      656
MAXssO       618
MINssO       618
MINsssN      613
MAXsssN      613
Length: 78, dtype: int64
Total borderline descriptors (50-90% NaN): 78


In [44]:
# Load the original dataset
import pandas as pd
all_sheets = pd.read_excel('..//Data/hob_data_set.xlsx', sheet_name=None)
train = all_sheets['Training set']

# Check the SMILES column name in both dataframes
print("descriptors_df SMILES column:", descriptors_df.columns[0])
print("train SMILES column:", train.columns[1])
print()

# Merge on SMILES
merged_df = descriptors_df.merge(train[['smile', 'label_cutoff_50%', 'label_cutoff_20%']], 
                                  left_on='SMILES', 
                                  right_on='smile', 
                                  how='inner')

print(f"Original descriptors_df: {descriptors_df.shape[0]} molecules")
print(f"After merge: {merged_df.shape[0]} molecules")

descriptors_df SMILES column: SMILES
train SMILES column: smile

Original descriptors_df: 1157 molecules
After merge: 1157 molecules


In [45]:
# Hold-out split. Every decision from here on that looks at the labels is taken
# on the training molecules only, so the test molecules stay unseen until the
# final evaluation.
from sklearn.model_selection import train_test_split

train_smiles, test_smiles = train_test_split(
    merged_df['SMILES'].values,
    test_size=0.20,
    stratify=merged_df['label_cutoff_50%'].values,
    random_state=RANDOM_STATE)

pd.Series(train_smiles, name='SMILES').to_csv('../Data/train_smiles.csv', index=False)
pd.Series(test_smiles, name='SMILES').to_csv('../Data/test_smiles.csv', index=False)

train_mask = merged_df['SMILES'].isin(train_smiles).values
print(f"Train molecules: {train_mask.sum()}")
print(f"Test molecules:  {(~train_mask).sum()}")

Train molecules: 925
Test molecules:  232


In [46]:
# correlation analysis with point biserial correlation for binary target and continuous descriptors
from scipy import stats

borderline_cols = borderline_nan.index.tolist()

# Replace NaN with 0 temporarily for correlation calculation
temp_df = merged_df.loc[train_mask, borderline_cols].fillna(0)
temp_df['target'] = merged_df.loc[train_mask, 'label_cutoff_50%'].values

# Calculate point biserial correlation for each descriptor
correlations = {}
for col in borderline_cols:
    corr, pvalue = stats.pointbiserialr(temp_df['target'], temp_df[col])
    correlations[col] = {'correlation': round(corr, 4), 'pvalue': round(pvalue, 4)}

corr_df = pd.DataFrame(correlations).T
corr_df = corr_df.reindex(corr_df['correlation'].abs().sort_values(ascending=False).index)
print(corr_df)

          correlation  pvalue
MDEO-12       -0.1586  0.0000
MINsCl         0.1559  0.0000
MAXsCl         0.1539  0.0000
MINssssN      -0.1526  0.0000
MAXssssN      -0.1526  0.0000
...               ...     ...
MINaaNH        0.0033  0.9199
MINtsC         0.0031  0.9243
MAXaaNH        0.0031  0.9251
MDEN-12        0.0030  0.9279
MAXtsC         0.0017  0.9586

[78 rows x 2 columns]


In [47]:
# Define groups to keep based on correlation strength and domain knowledge correlation 0.05-0.10, pvalue < 0.05
keep_borderline = corr_df[
    (corr_df['correlation'].abs() > 0.10) & 
    (corr_df['pvalue'] < 0.05)
].index.tolist()

# Everything else in borderline gets dropped
drop_borderline = [col for col in borderline_cols if col not in keep_borderline]

print(f"Keeping: {len(keep_borderline)} descriptors")
print(f"Dropping: {len(drop_borderline)} descriptors")

# Fill NaN with 0 for the ones we keep
descriptors_df[keep_borderline] = descriptors_df[keep_borderline].fillna(0)

# Drop the rest
descriptors_df = descriptors_df.drop(columns=drop_borderline)

print(f"\nRemaining descriptors: {descriptors_df.shape[1]}")

Keeping: 9 descriptors
Dropping: 69 descriptors

Remaining descriptors: 1682


## NaN in 20-50% of molecules

In [48]:
nan_summary_updated = descriptors_df.isnull().sum()
nan_summary_updated = nan_summary_updated[nan_summary_updated > 0]

investigate_20_50 = nan_summary_updated[
    (nan_summary_updated > len(descriptors_df) * 0.20) & 
    (nan_summary_updated <= len(descriptors_df) * 0.50)
]

print(f"Total descriptors in 20-50% NaN range: {len(investigate_20_50)}")
print(investigate_20_50)

Total descriptors in 20-50% NaN range: 14
MAXsCH3     295
MAXsssCH    411
MAXdssC     368
MAXsOH      388
MAXdO       439
MINsCH3     295
MINsssCH    411
MINdssC     368
MINsOH      388
MINdO       439
MDEC-11     570
MDEC-12     293
MDEC-13     287
MDEO-11     410
dtype: int64


In [49]:
from scipy import stats

cols_20_50 = investigate_20_50.index.tolist()
temp_df = merged_df.loc[train_mask, cols_20_50].fillna(0)
temp_df['target'] = merged_df.loc[train_mask, 'label_cutoff_50%'].values

correlations_20_50 = {}
for col in cols_20_50:
    corr, pvalue = stats.pointbiserialr(temp_df['target'], temp_df[col])
    correlations_20_50[col] = {'correlation': round(corr, 4), 'pvalue': round(pvalue, 4)}

corr_df_20_50 = pd.DataFrame(correlations_20_50).T
corr_df_20_50 = corr_df_20_50.reindex(corr_df_20_50['correlation'].abs().sort_values(ascending=False).index)
print(corr_df_20_50)

          correlation  pvalue
MDEO-11       -0.1813  0.0000
MDEC-12       -0.1770  0.0000
MINsssCH       0.1268  0.0001
MDEC-13       -0.1159  0.0004
MDEC-11       -0.1091  0.0009
MAXsOH        -0.0757  0.0214
MAXsCH3       -0.0752  0.0221
MAXdssC       -0.0618  0.0603
MINsOH        -0.0570  0.0833
MINsCH3       -0.0470  0.1532
MINdO          0.0351  0.2864
MAXdO          0.0250  0.4469
MINdssC       -0.0111  0.7371
MAXsssCH      -0.0063  0.8486


In [ ]:
# Keep only correlation > 0.10 and pvalue < 0.05
keep_20_50 = corr_df_20_50[
    (corr_df_20_50['correlation'].abs() > 0.10) & 
    (corr_df_20_50['pvalue'] < 0.05)
].index.tolist()

drop_20_50 = [col for col in cols_20_50 if col not in keep_20_50]

print(f"\nKeeping: {len(keep_20_50)} descriptors")
print(f"Dropping: {len(drop_20_50)} descriptors")

# Fill NaN with 0 for kept, drop the rest
descriptors_df[keep_20_50] = descriptors_df[keep_20_50].fillna(0)
descriptors_df = descriptors_df.drop(columns=drop_20_50, errors='ignore')

print(f"\nRemaining descriptors: {descriptors_df.shape[1]}")


Keeping: 5 descriptors
Dropping: 9 descriptors

Remaining descriptors: 1673


## NaN in 5-20% of molecules

In [51]:
# Check which descriptors are in 5-20% of molecules and see if they are important
nan_summary_updated = descriptors_df.isnull().sum()
nan_summary_updated = nan_summary_updated[nan_summary_updated > 0]

investigate = nan_summary_updated[(nan_summary_updated > len(descriptors_df)*0.05) & 
                                  (nan_summary_updated <= len(descriptors_df)*0.20)]
print(investigate)

MAXssCH2    149
MAXaaCH     203
MAXaasC     199
MINssCH2    149
MINaaCH     203
MINaasC     199
dtype: int64


In [52]:
from rdkit import Chem

# Verify that the NaN really means the atom type is absent, for every affected molecule.
cols_to_check = ['MAXaaCH', 'MINaaCH', 'MAXaasC', 'MINaasC', 'MINssCH2', 'MAXssCH2']

for col in cols_to_check:
    if col not in descriptors_df.columns:
        continue
    partner = 'S' + col[3:]          # MAXaaCH -> SaaCH, the summed version of the same atom type
    nan_mask = descriptors_df[col].isna()
    if not nan_mask.any():
        print(f"{col}: no NaN")
        continue
    if partner in descriptors_df.columns:
        # The summed descriptor is 0 (never NaN) when the atom type is absent.
        violations = (descriptors_df.loc[nan_mask, partner].fillna(0) != 0).sum()
        print(f"{col}: {nan_mask.sum()} NaN, {violations} of them with {partner} != 0")
    else:
        print(f"{col}: {nan_mask.sum()} NaN, no {partner} column to cross-check")

# Direct chemical check on the aromatic CH case
nan_mask = descriptors_df['MAXaaCH'].isna()
violations = 0
for smi in descriptors_df.loc[nan_mask, 'SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    aromatic_CH = [atom for atom in mol.GetAtoms()
                   if atom.GetIsAromatic()
                   and atom.GetSymbol() == 'C'
                   and atom.GetTotalNumHs() > 0]
    if len(aromatic_CH) > 0:
        violations += 1

print(f"\nMolecules with NaN in MAXaaCH that DO have aromatic CH atoms: {violations}")
print(f"Checked {nan_mask.sum()} molecules")

MAXaaCH: 203 NaN, 0 of them with SaaCH != 0
MINaaCH: 203 NaN, 0 of them with SaaCH != 0
MAXaasC: 199 NaN, 0 of them with SaasC != 0
MINaasC: 199 NaN, 0 of them with SaasC != 0
MINssCH2: 149 NaN, 0 of them with SssCH2 != 0
MAXssCH2: 149 NaN, 0 of them with SssCH2 != 0

Molecules with NaN in MAXaaCH that DO have aromatic CH atoms: 0
Checked 203 molecules


In [53]:
# Replace NaN values with 0 cause this atoms are not present in those molecules and the descriptor is counting them, so 0 is a reasonable value to impute
cols_to_fill = ['MAXaaCH', 'MINaaCH', 'MAXaasC', 'MINaasC', 'MINssCH2', 'MAXssCH2']
descriptors_df[cols_to_fill] = descriptors_df[cols_to_fill].fillna(0)

# Verify
print(descriptors_df[cols_to_fill].isnull().sum())

MAXaaCH     0
MINaaCH     0
MAXaasC     0
MINaasC     0
MINssCH2    0
MAXssCH2    0
dtype: int64


## NaN in <5% of molecules

In [ ]:
#Identify the <5% NaN descriptors
nan_summary_updated = descriptors_df.isnull().sum()
nan_summary_updated = nan_summary_updated[nan_summary_updated > 0]

investigate_5 = nan_summary_updated[nan_summary_updated <= len(descriptors_df) * 0.05]

print(f"Total descriptors with <5% NaN: {len(investigate_5)}")
print(investigate_5)

# Median imputation, medians learned from the training molecules only
from sklearn.impute import SimpleImputer

cols_5 = investigate_5.index.tolist()
train_rows = descriptors_df['SMILES'].isin(train_smiles).values

imputer = SimpleImputer(strategy='median')
imputer.fit(descriptors_df.loc[train_rows, cols_5])
descriptors_df[cols_5] = imputer.transform(descriptors_df[cols_5])

# Verify
print(f"\nRemaining NaN values in these columns: {descriptors_df[cols_5].isnull().sum().sum()}")
print(f"Remaining descriptors: {descriptors_df.shape[1]}")

Total descriptors with <5% NaN: 153
AATS6dv     6
AATS7dv    17
AATS8dv    45
AATS6d      6
AATS7d     17
           ..
AXp-7dv    27
MDEC-22    34
MDEC-23    26
MDEC-33    25
Vabc        6
Length: 153, dtype: int64

Remaining NaN values in these columns: 0
Remaining descriptors: 1673


In [55]:
print(f"Total NaN values remaining: {descriptors_df.isnull().sum().sum()}")

Total NaN values remaining: 0


In [56]:
# Save the cleaned and processed descriptors to a new CSV file
descriptors_df.to_csv('../Data/mordred_descriptors_without_NaN.csv', index=False)

In [57]:
descriptors_without_NaN = pd.read_csv('../Data/mordred_descriptors_without_NaN.csv')

In [58]:
# Prepare the feature matrix X and target vector y for machine learning
labels_by_smiles = merged_df.set_index('SMILES')[['label_cutoff_50%', 'label_cutoff_20%']]

X = descriptors_df.drop(columns=["SMILES"])
y_50 = labels_by_smiles.loc[descriptors_df['SMILES'], 'label_cutoff_50%'].values
y_20 = labels_by_smiles.loc[descriptors_df['SMILES'], 'label_cutoff_20%'].values

print(X.shape, y_50.shape, y_20.shape)

(1157, 1672) (1157,) (1157,)


In [59]:
# Check for columns with zero variance and drop them
zero_var_cols = [col for col in X.columns if X[col].std() == 0]
for col in zero_var_cols:
    print(f"Column '{col}' has zero variance and will be dropped.")
X = X.drop(columns=zero_var_cols)
print(f"Total columns dropped: {len(zero_var_cols)}")

Column 'SM1_Dt' has zero variance and will be dropped.
Column 'NsLi' has zero variance and will be dropped.
Column 'NssBe' has zero variance and will be dropped.
Column 'NssssBe' has zero variance and will be dropped.
Column 'NssBH' has zero variance and will be dropped.
Column 'NssssB' has zero variance and will be dropped.
Column 'NddC' has zero variance and will be dropped.
Column 'NsNH3' has zero variance and will be dropped.
Column 'NsSiH3' has zero variance and will be dropped.
Column 'NssSiH2' has zero variance and will be dropped.
Column 'NsssSiH' has zero variance and will be dropped.
Column 'NssssSi' has zero variance and will be dropped.
Column 'NsPH2' has zero variance and will be dropped.
Column 'NssPH' has zero variance and will be dropped.
Column 'NsssP' has zero variance and will be dropped.
Column 'NsssssP' has zero variance and will be dropped.
Column 'NsGeH3' has zero variance and will be dropped.
Column 'NssGeH2' has zero variance and will be dropped.
Column 'NsssGe

## Check for Outliers

In [60]:
# Check how many descriptors have extreme outliers
from scipy import stats
import numpy as np

numeric = descriptors_without_NaN.select_dtypes(include=[np.number])
numeric = numeric.loc[:, numeric.std() > 0]

z_scores = pd.DataFrame(stats.zscore(numeric), columns=numeric.columns)

# Count descriptors where any molecule has |z-score| > 3
outlier_cols = (z_scores.abs() > 3).any(axis=0).sum()
outlier_mols = (z_scores.abs() > 3).any(axis=1).sum()

print(f"Descriptors with at least one outlier (|z| > 3): {outlier_cols}")
print(f"Molecules with at least one outlier value: {outlier_mols}")

Descriptors with at least one outlier (|z| > 3): 1518
Molecules with at least one outlier value: 1002
